(how_to_choose_turbine_label)=

# Choose a turbine model

A turbine model needs an efficiency specification as well as the inlet and
outlet conditions. Both TESPy turbine classes can expand steam: the extra
`SteamTurbine` class accounts for efficiency loss during wet expansion.

| Model or specification | Data you need | Appropriate use |
| --- | --- | --- |
| {py:class}`~tespy.components.turbomachinery.turbine.Turbine` with `eta_s` | Overall isentropic efficiency at the operating point | A gas or steam expansion when the overall efficiency is known |
| `Turbine` with `eta_s_char` | A design point and an efficiency characteristic | An empirical offdesign efficiency variation |
| {py:class}`~tespy.components.turbomachinery.steam_turbine.SteamTurbine` with `eta_s_dry` and `alpha` | Dry isentropic efficiency and a wetness correction factor | A pure-water expansion where liquid formation should reduce efficiency |

`SteamTurbine` requires a pure fluid and is intended for water. Its Baumann
correlation corrects the efficiency of the wet part of an expansion; it is not
a blade-resolved model. It also inherits the standard turbine parameters.
Choose either the overall `eta_s` or the dry-efficiency/wetness specification,
rather than imposing both.

A fixed overall efficiency may already include wetness losses in the
measurement used to obtain it. Do not apply the wetness penalty a second
time to that value. More detailed modeling is useful only when its inputs
and assumptions suit the machine.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from tespy.components import Sink, Source, SteamTurbine, Turbine
from tespy.connections import Connection
from tespy.networks import Network

## Compare the same steam expansion

Use 10 kg/s of water at 20 bar and 250 °C, expanding to 0.1 bar.
The standard turbine holds the overall efficiency at 0.9; the wet-expansion
model uses a dry efficiency of 0.9 and `alpha=1`. This illustrates the
effect of the correlation, not two fitted descriptions of a specific machine.

Every calculation below checks solver convergence and the energy balance.
TESPy reports turbine power as negative because energy leaves the fluid;
the tables and plots show its positive output magnitude in MW.

In [ ]:
def expand_steam(model, outlet_pressure=0.1, inlet_temperature=250, **parameters):
    network = Network(iterinfo=False)
    network.units.set_defaults(
        temperature="degC", pressure="bar",
        pressure_difference="bar", power="MW",
    )
    turbine = model("turbine")
    inlet = Connection(Source("source"), "out1", turbine, "in1")
    outlet = Connection(turbine, "out1", Sink("sink"), "in1")
    network.add_conns(inlet, outlet)
    inlet.set_attr(fluid={"water": 1}, m=10, p=20, T=inlet_temperature)
    outlet.set_attr(p=outlet_pressure)
    turbine.set_attr(**parameters)
    network.solve("design")
    network.assert_convergence()

    expected_power = inlet.m.val_SI * (outlet.h.val_SI - inlet.h.val_SI)
    assert np.isclose(turbine.P.val_SI, expected_power, rtol=1e-9)
    assert np.isclose(inlet.m.val_SI, outlet.m.val_SI)
    assert 0 < turbine.eta_s.val_SI <= 1
    assert turbine.P.val_SI < 0
    phase = outlet.calc_phase()
    quality = outlet.x.val if phase == "tp" else np.nan
    if phase == "tp":
        assert 0 <= quality <= 1
    return {
        "outlet pressure / bar": outlet_pressure,
        "efficiency": turbine.eta_s.val_SI,
        "power output / MW": -turbine.P.val,
        "outlet phase": phase,
        "vapor mass fraction": quality,
    }


models = {
    "Fixed overall efficiency": (Turbine, {"eta_s": 0.9}),
    "Wetness correction": (SteamTurbine, {"eta_s_dry": 0.9, "alpha": 1.0}),
}

In [ ]:
baseline = pd.DataFrame({
    name: expand_steam(model, **parameters)
    for name, (model, parameters) in models.items()
}).T
display(baseline.round(3))

fixed = baseline.loc["Fixed overall efficiency"]
wet = baseline.loc["Wetness correction"]
assert wet["efficiency"] < fixed["efficiency"]
assert wet["power output / MW"] < fixed["power output / MW"]
assert wet["vapor mass fraction"] > fixed["vapor mass fraction"]

At this operating point both outlets are wet. Applying the correction
reduces the calculated efficiency and power output. The outlet is drier
because less enthalpy was extracted, even though the wetness model is the one
accounting for a loss. Vapor mass fraction is meaningful only for a
two-phase state; the following table leaves it blank for superheated steam.

## Vary the back pressure

Now keep the inlet state and mass flow fixed while changing the outlet
pressure. Each point is a separate design calculation: this sweep isolates
the wetness correction and does not apply an offdesign cone law or an
efficiency characteristic.

A real machine's operating envelope, flow capacity and efficiency may impose
additional restrictions. Use measured efficiencies or calibrated
characteristics when predicting that machine.

In [ ]:
pressures = np.geomspace(0.1, 10, 25)
rows = []
for name, (model, parameters) in models.items():
    for pressure in pressures:
        row = expand_steam(model, outlet_pressure=float(pressure), **parameters)
        row["model"] = name
        rows.append(row)
results = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
for name, values in results.groupby("model", sort=False):
    axes[0].plot(values["outlet pressure / bar"], values["efficiency"], label=name)
    axes[1].plot(values["outlet pressure / bar"], values["power output / MW"], label=name)
for axis in axes:
    axis.set_xscale("log")
    axis.set_xlabel("Outlet pressure / bar")
    axis.grid(True, alpha=0.3)
axes[0].set_ylabel("Overall isentropic efficiency")
axes[1].set_ylabel("Power output / MW")
axes[0].legend()
plt.show()

In [ ]:
selected = results.groupby("model", sort=False).nth([0, 8, 16, 24])
display(selected.round(3))

## Check a dry expansion

At an inlet temperature of 500 °C and an outlet pressure of 1 bar, both
models remain superheated throughout the expansion. With the same
efficiency input, they should then agree.
This control also shows that choosing the `SteamTurbine` class does not
always imply an additional loss.

In [ ]:
dry = pd.DataFrame({
    name: expand_steam(model, inlet_temperature=500, outlet_pressure=1, **parameters)
    for name, (model, parameters) in models.items()
}).T
assert (dry["outlet phase"] == "g").all()
np.testing.assert_allclose(dry["efficiency"].astype(float), 0.9, rtol=1e-8)
np.testing.assert_allclose(
    dry["power output / MW"].astype(float),
    float(dry.iloc[0]["power output / MW"]),
    rtol=1e-8,
)
display(dry.round(3))

## Choose the parameters for your task

- Use `Turbine(eta_s=...)` when an overall efficiency is known. It can
  represent a steam turbine without separately predicting wetness losses.
- Use the standard turbine's `eta_s_char` for a measured part-load
  relationship. Its design point and normalized characteristic are separate
  from the wetness correction demonstrated here.
- Use `SteamTurbine(eta_s_dry=..., alpha=...)` when dry efficiency and an
  appropriate wetness factor are available. The Baumann rule handles an
  expansion that starts dry and enters the vapor dome as well as a wet inlet.
- The inherited `cone` relation describes the offdesign relationship between
  pressure, inlet state and mass flow; it does not replace an efficiency
  model.

See the {py:class}`~tespy.components.turbomachinery.turbine.Turbine` and
{py:class}`~tespy.components.turbomachinery.steam_turbine.SteamTurbine` API
examples for the complete parameter lists and design/offdesign setup.
The wetness model implementation also documents its Baumann-rule reference.